# Sparse table

Responde consultas sobre rangos de un array que no cambia. El nivel $k$ guarda el resultado de cada bloque de longitud $2^k$:

$$T_{k,i} = T_{k-1,i} \oplus T_{k-1,\,i + 2^{k-1}}$$

Construirla cuesta $O(n \log n)$ en tiempo y memoria. Si la operación es idempotente ($x \oplus x = x$, como el mínimo, el máximo o el mcd), cualquier rango $[l, r)$ se cubre con dos bloques solapados de longitud $2^k$, $k = \lfloor \log_2(r - l) \rfloor$, y la consulta cuesta $O(1)$:

$$\operatorname{op}[l, r) = T_{k,l} \oplus T_{k,\,r - 2^k}$$

Para operaciones no idempotentes, como la suma, los bloques no pueden solaparse y la consulta pasa a $O(\log n)$.

## 1. Construir la tabla de mínimos

Construir la sparse table de mínimos de las alturas de las cotas de un sendero, nivel a nivel, y mostrar cada nivel. Comprobar que cada entrada $T_{k,i}$ es el mínimo de su bloque.

In [1]:
import math
import operator
import random
from functools import reduce


def construir(a, op=min):
    tabla = [list(a)]
    k = 1
    while 1 << k <= len(a):
        anterior, mitad = tabla[-1], 1 << (k - 1)
        tabla.append([op(anterior[i], anterior[i + mitad]) for i in range(len(a) - (1 << k) + 1)])
        k += 1
    return tabla


cotas = [820, 760, 905, 640, 710, 980, 590, 870, 930, 650]
tabla = construir(cotas)
for k, nivel in enumerate(tabla):
    print(f"bloques de {1 << k:>2}:", nivel)

for k, nivel in enumerate(tabla):
    for i, valor in enumerate(nivel):
        assert valor == min(cotas[i:i + (1 << k)])

bloques de  1: [820, 760, 905, 640, 710, 980, 590, 870, 930, 650]
bloques de  2: [760, 760, 640, 640, 710, 590, 590, 870, 650]
bloques de  4: [640, 640, 640, 590, 590, 590, 590]
bloques de  8: [590, 590, 590]


## 2. Consultas en O(1) con dos bloques solapados

Escribir `consultar(tabla, l, r, op)` para el rango $[l, r)$ usando $k = $ `(r - l).bit_length() - 1` (que es $\lfloor \log_2 (r-l) \rfloor$) y los dos bloques que empiezan en $l$ y terminan en $r$. Mostrar qué bloques usa una consulta y comprobar miles de consultas aleatorias contra `min` del corte.

In [2]:
def consultar(tabla, l, r, op=min):
    k = (r - l).bit_length() - 1
    return op(tabla[k][l], tabla[k][r - (1 << k)])


l, r = 2, 9
k = (r - l).bit_length() - 1
print(f"[{l}, {r}): bloques [{l}, {l + (1 << k)}) y [{r - (1 << k)}, {r})")
print("mínimo:", consultar(tabla, l, r), "=", min(cotas[l:r]))

random.seed(4)
a = [random.randint(-1000, 1000) for _ in range(500)]
t = construir(a)
for _ in range(5000):
    l, r = sorted(random.sample(range(len(a) + 1), 2))
    assert consultar(t, l, r) == min(a[l:r])
assert all((m).bit_length() - 1 == math.floor(math.log2(m)) for m in range(1, 10_000))

[2, 9): bloques [2, 6) y [5, 9)
mínimo: 590 = 590


## 3. Cualquier operación idempotente

La misma estructura sirve para el máximo, el mcd, el AND y el OR de bits, porque repetir un elemento no cambia el resultado. Construir una tabla para cada operación sobre el mismo array y comprobarlas contra `reduce` sobre el corte. Calcular, por ejemplo, el mcd de los lotes de cajas de papas de varios tramos de pedidos.

In [3]:
operaciones = {"mín": min, "máx": max, "mcd": math.gcd, "and": operator.and_, "or": operator.or_}
lotes = [36, 48, 60, 24, 84, 120, 90, 45, 30, 75]
tablas = {nombre: construir(lotes, op) for nombre, op in operaciones.items()}
for l, r in [(0, 4), (3, 7), (5, 10)]:
    print(f"[{l}, {r})", lotes[l:r], {nombre: consultar(tablas[nombre], l, r, op) for nombre, op in operaciones.items()})

for _ in range(300):
    a = [random.randint(1, 200) for _ in range(random.randint(1, 40))]
    for op in operaciones.values():
        t = construir(a, op)
        for _ in range(20):
            l, r = sorted(random.sample(range(len(a) + 1), 2))
            assert consultar(t, l, r, op) == reduce(op, a[l:r])

[0, 4) [36, 48, 60, 24] {'mín': 24, 'máx': 60, 'mcd': 12, 'and': 0, 'or': 60}
[3, 7) [24, 84, 120, 90] {'mín': 24, 'máx': 120, 'mcd': 6, 'and': 16, 'or': 126}
[5, 10) [120, 90, 45, 30, 75] {'mín': 30, 'máx': 120, 'mcd': 15, 'and': 8, 'or': 127}


## 4. La suma no admite solapes

Comprobar que con la suma la consulta de dos bloques solapados falla (cuenta dos veces la intersección) y escribir una consulta en $O(\log n)$ que descompone el rango en bloques disjuntos de potencias de dos, recorriendo los bits de la longitud de mayor a menor.

In [4]:
def consultar_disjunto(tabla, l, r, op, neutro):
    resultado = neutro
    for k in range(len(tabla) - 1, -1, -1):
        if l + (1 << k) <= r:
            resultado = op(resultado, tabla[k][l])
            l += 1 << k
    return resultado


ventas = [5, 3, 8, 6, 2, 7, 4, 1]
t_suma = construir(ventas, operator.add)
print("Solapada:", consultar(t_suma, 1, 7, operator.add), " disjunta:", consultar_disjunto(t_suma, 1, 7, operator.add, 0), " real:", sum(ventas[1:7]))

for _ in range(500):
    a = [random.randint(-50, 50) for _ in range(random.randint(1, 60))]
    t = construir(a, operator.add)
    l, r = sorted(random.sample(range(len(a) + 1), 2))
    assert consultar_disjunto(t, l, r, operator.add, 0) == sum(a[l:r])
    assert consultar_disjunto(construir(a, min), l, r, min, math.inf) == min(a[l:r])

Solapada: 38  disjunta: 30  real: 30


## 5. Posición del mínimo

A veces interesa dónde está el mínimo, no su valor. Guardar en la tabla índices y comparar por `(valor, índice)` para desempatar con la primera aparición. Comprobarlo contra `min(range(l, r), key=...)`.

In [5]:
def construir_posiciones(a):
    def mejor(i, j):
        return min(i, j, key=lambda x: (a[x], x))

    return construir(range(len(a)), mejor), mejor


temperaturas = [18, 16, 17, 16, 19, 15, 15, 20]
t_pos, mejor = construir_posiciones(temperaturas)
for l, r in [(0, 4), (2, 8), (5, 7)]:
    i = consultar(t_pos, l, r, mejor)
    print(f"[{l}, {r}): mínimo {temperaturas[i]} en la posición {i}")

for _ in range(500):
    a = [random.randint(0, 5) for _ in range(random.randint(1, 30))]
    t, f = construir_posiciones(a)
    l, r = sorted(random.sample(range(len(a) + 1), 2))
    assert consultar(t, l, r, f) == min(range(l, r), key=lambda x: a[x])

[0, 4): mínimo 16 en la posición 1
[2, 8): mínimo 15 en la posición 5
[5, 7): mínimo 15 en la posición 5


## 6. Tamaño de la tabla

Calcular cuántas entradas guarda la tabla para varios $n$ y compararlo con $n \lfloor \log_2 n \rfloor + n$ y con las $\binom{n+1}{2}$ respuestas que habría que guardar para precalcular todos los rangos.

In [6]:
print(f"{'n':>7} {'entradas':>10} {'n·log n + n':>12} {'todos los rangos':>17}")
for n in (10, 100, 1000, 10_000):
    entradas = sum(len(nivel) for nivel in construir(range(n), min))
    cota = n * int(math.log2(n)) + n
    print(f"{n:>7} {entradas:>10} {cota:>12} {math.comb(n + 1, 2):>17}")
    assert entradas <= cota

      n   entradas  n·log n + n  todos los rangos
     10         29           40                55
    100        580          700              5050
   1000       8987        10000            500500
  10000     123631       140000          50005000


## 7. Frente a un árbol de segmentos

Escribir un árbol de segmentos iterativo de mínimos (consulta en $O(\log n)$, pero admite actualizaciones) y comprobar que ambos dan lo mismo en muchas consultas. Contar cuántas veces se aplica la operación por consulta en cada estructura: la sparse table siempre usa una, el árbol de segmentos crece con $\log n$.

In [7]:
class ArbolSegmentos:
    def __init__(self, a, op=min):
        self.n, self.op = len(a), op
        self.t = [None] * self.n + list(a)
        for i in range(self.n - 1, 0, -1):
            self.t[i] = op(self.t[2 * i], self.t[2 * i + 1])

    def consultar(self, l, r):
        resultado, self.usos = None, 0
        l += self.n
        r += self.n
        while l < r:
            if l & 1:
                resultado = self.t[l] if resultado is None else self.op(resultado, self.t[l])
                self.usos += 1
                l += 1
            if r & 1:
                r -= 1
                resultado = self.t[r] if resultado is None else self.op(resultado, self.t[r])
                self.usos += 1
            l //= 2
            r //= 2
        return resultado


print(f"{'n':>7} {'usos máx. árbol':>16} {'2·log2 n':>9}")
for n in (16, 256, 4096, 65_536):
    a = [random.randint(0, 10**6) for _ in range(n)]
    t, arbol = construir(a), ArbolSegmentos(a)
    usos = 0
    for _ in range(2000):
        l, r = sorted(random.sample(range(n + 1), 2))
        assert consultar(t, l, r) == arbol.consultar(l, r)
        usos = max(usos, arbol.usos)
    print(f"{n:>7} {usos:>16} {2 * int(math.log2(n)):>9}")
    assert usos <= 2 * math.log2(n)

      n  usos máx. árbol  2·log2 n
     16                6         8
    256               12        16
   4096               17        24


  65536               23        32


## 8. Desnivel de cada etapa

Un sendero tiene la altitud medida cada 100 m. Para cada etapa $[l, r)$ de un plan de ruta, calcular el desnivel máximo (altitud máxima menos mínima) con dos sparse tables, y la etapa de mayor desnivel. Comprobarlo contra `max - min` del corte.

In [8]:
random.seed(11)
altitud = [600]
for _ in range(199):
    altitud.append(max(0, altitud[-1] + random.randint(-40, 45)))
t_min, t_max = construir(altitud, min), construir(altitud, max)

etapas = [(0, 40), (40, 95), (95, 130), (130, 170), (170, 200)]
print(f"{'etapa':>11} {'mín':>6} {'máx':>6} {'desnivel':>9}")
for l, r in etapas:
    bajo, alto = consultar(t_min, l, r, min), consultar(t_max, l, r, max)
    print(f"{f'[{l}, {r})':>11} {bajo:>6} {alto:>6} {alto - bajo:>9}")
    assert alto - bajo == max(altitud[l:r]) - min(altitud[l:r])
peor = max(etapas, key=lambda e: consultar(t_max, *e, max) - consultar(t_min, *e, min))
print("Etapa más dura:", peor)

      etapa    mín    máx  desnivel
    [0, 40)    600    993       393
   [40, 95)    746   1057       311
  [95, 130)    646    774       128
 [130, 170)    587    877       290
 [170, 200)    736    925       189
Etapa más dura: (0, 40)


## 9. Tramo más largo con poca variación

Encontrar el tramo contiguo más largo de la serie anterior en el que la altitud no varía más de $d$ metros (máximo menos mínimo $\le d$). Con dos punteros y las consultas en $O(1)$ de las dos tablas, el coste es $O(n)$ tras construirlas. Comprobarlo contra la fuerza bruta.

In [9]:
def tramo_mas_largo(a, d):
    t_min, t_max = construir(a, min), construir(a, max)
    mejor, l = (0, 0), 0
    for r in range(1, len(a) + 1):
        while consultar(t_max, l, r, max) - consultar(t_min, l, r, min) > d:
            l += 1
        mejor = max(mejor, (r - l, -l))
    return -mejor[1], -mejor[1] + mejor[0]


def tramo_mas_largo_bruto(a, d):
    return max(((j - i, -i) for i in range(len(a)) for j in range(i + 1, len(a) + 1) if max(a[i:j]) - min(a[i:j]) <= d), default=(0, 0))


for d in (20, 50, 100):
    l, r = tramo_mas_largo(altitud, d)
    print(f"d = {d:>3}: tramo [{l}, {r}) de {r - l} medidas")

for _ in range(500):
    a = [random.randint(0, 20) for _ in range(random.randint(1, 25))]
    d = random.randint(0, 10)
    l, r = tramo_mas_largo(a, d)
    largo, menos_l = tramo_mas_largo_bruto(a, d)
    assert (r - l, -l) == (largo, menos_l)

d =  20: tramo [176, 182) de 6 medidas
d =  50: tramo [151, 165) de 14 medidas
d = 100: tramo [98, 124) de 26 medidas


## 10. Subarrays con un mcd dado

Contar los subarrays cuyo mcd es exactamente $g$. Fijado el inicio $l$, el mcd de $[l, r)$ no crece al alargar el tramo, y cada valor nuevo divide al anterior. Por eso las condiciones "mcd $\le g$" y "mcd no es múltiplo de $g$" son monótonas en $r$: con la sparse table de mcd y dos búsquedas binarias se obtiene el intervalo de finales en que el mcd vale exactamente $g$. Son $O(n \log n)$ consultas. Comprobarlo contra la fuerza bruta.

In [10]:
def primer_fin(t, l, n, condicion):
    """Menor r en (l, n] con condicion(mcd[l, r)), suponiendo que es monótona; n + 1 si no hay."""
    bajo, alto = l + 1, n + 1
    while bajo < alto:
        medio = (bajo + alto) // 2
        if condicion(consultar(t, l, medio, math.gcd)):
            alto = medio
        else:
            bajo = medio + 1
    return bajo


def subarrays_con_mcd(a, g):
    n, t, total = len(a), construir(a, math.gcd), 0
    for l in range(n):
        desde = primer_fin(t, l, n, lambda x: x <= g)
        hasta = primer_fin(t, l, n, lambda x: x % g != 0)
        total += max(0, hasta - desde)
    return total


def subarrays_con_mcd_bruto(a, g):
    return sum(reduce(math.gcd, a[i:j]) == g for i in range(len(a)) for j in range(i + 1, len(a) + 1))


cajas = [12, 18, 6, 9, 3, 24, 36, 4]
for g in (1, 3, 6, 12):
    print(f"Subarrays con mcd {g:>2}: {subarrays_con_mcd(cajas, g)}")

for _ in range(500):
    a = [random.choice([2, 3, 4, 6, 8, 9, 12, 18, 24]) for _ in range(random.randint(1, 15))]
    g = random.choice([1, 2, 3, 6])
    assert subarrays_con_mcd(a, g) == subarrays_con_mcd_bruto(a, g)

Subarrays con mcd  1: 5
Subarrays con mcd  3: 18
Subarrays con mcd  6: 4
Subarrays con mcd 12: 2
